# Step 4. Clustering Model Selection

## 목적
Step 3.5에서 생성한 행동 축 기반 데이터를 사용해 여러 clustering 후보를 비교하고 최종 segment model을 확정한다.

## 입력 파일
- `data/processed/BankChurners_cluster_input.csv`
- `data/processed/BankChurners_profile_table.csv`
- `data/processed/cluster_feature_info.json`

## 출력 파일
- `data/analysis/clustering_results/09_selected_model.json`
- `data/analysis/clustering_results/13_final_customer_segments_core7_kmeans_k5.csv`
- k=4, k=5, k=6 review shortlist 파일

## 전체 프로젝트에서의 역할
이 단계는 최종 clustering model과 segment label을 만드는 핵심 모델 선택 단계이다. 이후 Step 4.5, Step 5, Step 6/7은 이 label을 기준으로 진행된다.

## 주요 분석 로직
Silhouette score, Davies-Bouldin index, Calinski-Harabasz score 같은 내부 지표와 cluster size balance, segment별 이탈률 차이, high-risk signal 차이를 함께 검토한다. k=4, k=5, k=6 후보는 해석 가능성까지 비교한다.

## 결과 해석 시 주의점
자동 ranking score는 모델 선택을 돕는 기준이며 유일한 판단 근거가 아니다. 최종 모델은 해석 가능성, segment 크기 균형, 이탈률 차이, high-risk signal 차이를 종합해 `core_7_KMeans_k5`로 고정한다.

In [1]:
from pathlib import Path
import sys
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import joblib

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current] + list(current.parents):
        if (path / "data").exists() and (path / "src").exists():
            return path
    return current


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src.common import read_csv_safely, save_csv, save_json

# ============================================================
# Reproducibility settings
# ============================================================
# 기존 실험 결과의 cluster 번호와 label을 최대한 유지하기 위해
# 기존 KMeans n_init=30은 그대로 둔다.
# n_init을 50 등으로 바꾸면 결과가 더 안정적일 수는 있지만,
# cluster 번호/centroid가 달라질 수 있으므로 반드시 Step4.5를 다시 확인해야 한다.
RANDOM_STATE = 42
KMEANS_N_INIT = 30
GMM_N_INIT = 5

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

REPRODUCIBILITY_CONFIG = {
    "random_state": RANDOM_STATE,
    "pythonhashseed": os.environ.get("PYTHONHASHSEED"),
    "kmeans_n_init": KMEANS_N_INIT,
    "gmm_n_init": GMM_N_INIT,
    "note": (
        "PYTHONHASHSEED is most effective when set before the Python process starts. "
        "random.seed and np.random.seed are set for this notebook run."
    ),
}

print("Project root:", ROOT)
print("Reproducibility config:", REPRODUCIBILITY_CONFIG)


Project root: C:\Users\user\Desktop\BankChurners-Customer-Segmentation
Reproducibility config: {'random_state': 42, 'pythonhashseed': '42', 'kmeans_n_init': 30, 'gmm_n_init': 5, 'note': 'PYTHONHASHSEED is most effective when set before the Python process starts. random.seed and np.random.seed are set for this notebook run.'}


## 1. 데이터 로드

Step 4의 입력은 Step 3.5에서 생성한 clustering input과 profiling table이다.

In [2]:
processed_dir = ROOT / "data" / "processed"
output_dir = ROOT / "data" / "analysis" / "clustering_results"
output_dir.mkdir(parents=True, exist_ok=True)

cluster_path = processed_dir / "BankChurners_cluster_input.csv"
profile_path = processed_dir / "BankChurners_profile_table.csv"
feature_info_path = processed_dir / "cluster_feature_info.json"

for path in [cluster_path, profile_path, feature_info_path]:
    if not path.exists():
        raise FileNotFoundError(f"필수 파일이 없습니다: {path}")

df_cluster = read_csv_safely(cluster_path)
df_profile = read_csv_safely(profile_path)
with open(feature_info_path, "r", encoding="utf-8") as f:
    feature_info = json.load(f)

if len(df_cluster) != len(df_profile):
    raise ValueError("cluster input과 profile table의 행 수가 다릅니다.")
if not df_cluster["Row_ID"].equals(df_profile["Row_ID"]):
    raise ValueError("Row_ID 정렬이 일치하지 않습니다.")

feature_sets = feature_info["feature_sets"]
print("df_cluster:", df_cluster.shape)
print("df_profile:", df_profile.shape)
print("feature sets:", list(feature_sets.keys()))

df_cluster: (10127, 9)
df_profile: (10127, 59)
feature sets: ['core_7', 'core_7_plus_capacity']


## 2. Clustering 후보 비교 함수

동일한 평가 함수를 사용하여 feature set, 알고리즘, cluster 수를 비교한다. Silhouette은 실행 시간을 줄이기 위해 표본 기반으로 계산한다.

In [3]:
def evaluate_clustering(X: pd.DataFrame, labels: np.ndarray) -> dict:
    n_clusters = len(np.unique(labels))

    if n_clusters < 2:
        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan
        }

    return {
        "silhouette": silhouette_score(X, labels), # 높을수록 좋음
        "davies_bouldin": davies_bouldin_score(X, labels), # 낮을수록 좋음
        "calinski_harabasz": calinski_harabasz_score(X, labels), # 높을수록 좋음 
    }


def cluster_size_info(labels: np.ndarray) -> dict:
    counts = pd.Series(labels).value_counts().sort_index()
    ratios = counts / counts.sum()
    return {
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "min_cluster_ratio": float(ratios.min()),
        "max_cluster_ratio": float(ratios.max()),
    }


def evaluate_profile(df_profile: pd.DataFrame, labels: np.ndarray) -> tuple[dict, pd.DataFrame]:
    temp = df_profile.copy()
    temp["Cluster"] = labels
    summary = temp.groupby("Cluster").agg(
        n_customers=("Attrition_Flag", "size"),
        attrition_rate=("Attrition_Flag", "mean"),
        high_risk_signal_mean=("High_Risk_Signal_Count", "mean"),
    ).reset_index()
    metrics = {
        "attrition_rate_range": float(summary["attrition_rate"].max() - summary["attrition_rate"].min()),
        "high_risk_signal_range": float(summary["high_risk_signal_mean"].max() - summary["high_risk_signal_mean"].min()),
    }
    return metrics, summary


def fit_predict_model(algorithm: str, X: pd.DataFrame, k: int):
    if algorithm == "KMeans":
        # random_state와 n_init을 명시하여 같은 데이터/환경에서 재현 가능하게 한다.
        # n_init은 기존 결과 보존을 위해 KMEANS_N_INIT=30으로 유지한다.
        model = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=KMEANS_N_INIT
        )
        labels = model.fit_predict(X)
    elif algorithm == "GMM":
        model = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=RANDOM_STATE,
            n_init=GMM_N_INIT
        )
        labels = model.fit_predict(X)
    elif algorithm == "Agglomerative_Ward":
        # Agglomerative clustering은 deterministic한 알고리즘이므로 random_state가 없다.
        model = AgglomerativeClustering(n_clusters=k, linkage="ward")
        labels = model.fit_predict(X)
    else:
        raise ValueError(f"지원하지 않는 알고리즘: {algorithm}")
    return model, labels


## 3. 후보 모델 실행

핵심 행동 축 7개와 신용한도 규모를 추가한 후보 feature set을 비교한다. Cluster 수는 4~6개 범위에서 탐색한다.

In [4]:
algorithms = ["KMeans", "GMM", "Agglomerative_Ward"]
k_values = [4, 5, 6]

results = []
all_labels = {}
profile_summaries = []
trained_models = {}

for feature_set_name, cols in feature_sets.items():
    X = df_cluster[cols].copy()
    for k in k_values:
        for algorithm in algorithms:
            model, labels = fit_predict_model(algorithm, X, k)
            key = f"{feature_set_name}_{algorithm}_k{k}"
            all_labels[key] = labels
            if algorithm in ["KMeans", "GMM"]:
                trained_models[key] = model

            internal_metrics = evaluate_clustering(X, labels)
            size_metrics = cluster_size_info(labels)
            profile_metrics, profile_summary = evaluate_profile(df_profile, labels)
            profile_summary["candidate_key"] = key
            profile_summaries.append(profile_summary)

            results.append({
                "key": key,
                "feature_set": feature_set_name,
                "algorithm": algorithm,
                "k": k,
                **internal_metrics,
                **size_metrics,
                **profile_metrics,
            })

result_df = pd.DataFrame(results)
profile_summary_long = pd.concat(profile_summaries, ignore_index=True)

save_csv(result_df, output_dir / "01_clustering_candidate_metrics.csv")
save_csv(profile_summary_long, output_dir / "02_candidate_profile_summary_long.csv")

display(result_df.sort_values(["silhouette", "attrition_rate_range"], ascending=[False, False]).head(15))

,key,feature_set,algorithm,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_ratio,max_cluster_ratio,attrition_rate_range,high_risk_signal_range
0,core_7_KMeans_k4,core_7,KMeans,4,0.155258,1.814115,1545.182352,1938,3218,0.191370,0.317764,0.266996,0.976676
9,core_7_plus_capacity_KMeans_k4,core_7_plus_capacity,KMeans,4,0.150722,1.867298,1491.606070,1830,3151,0.180705,0.311148,0.267673,0.927550
3,core_7_KMeans_k5,core_7,KMeans,5,0.144931,1.837940,1441.379165,998,2373,0.098548,0.234324,0.410423,1.304520
12,core_7_plus_capacity_KMeans_k5,core_7_plus_capacity,KMeans,5,0.144295,1.933031,1364.892013,937,2706,0.092525,0.267206,0.465298,1.414689
15,core_7_plus_capacity_KMeans_k6,core_7_plus_capacity,KMeans,6,0.139443,1.866946,1251.528426,836,2461,0.082552,0.243014,0.498428,1.526204
6,core_7_KMeans_k6,core_7,KMeans,6,0.138869,1.724660,1346.529889,870,2069,0.085909,0.204305,0.470174,1.470179
2,core_7_Agglomerative_Ward_k4,core_7,Agglomerative_Ward,4,0.108716,2.119646,1090.625676,843,4777,0.083243,0.471709,0.197893,0.722769
8,core_7_Agglomerative_Ward_k6,core_7,Agglomerative_Ward,6,0.103883,1.978442,1005.221859,306,2986,0.030216,0.294855,0.449271,1.255553
11,core_7_plus_capacity_Agglomerative_Ward_k4,core_7_plus_capacity,Agglomerative_Ward,4,0.100880,2.090887,1072.116907,971,4682,0.095882,0.462328,0.220773,0.713541
14,core_7_plus_capacity_Agglomerative_Ward_k5,core_7_plus_capacity,Agglomerative_Ward,5,0.092854,2.300684,978.449360,971,3266,0.095882,0.322504,0.500749,1.476490


## 4. 최종 모델 선택

최종 모델은 내부 군집 품질, cluster size balance, 사후 해석 가능성을 함께 고려하여 선택한다. 아래 선택 점수는 재현 가능한 기본 기준이며, 보고서 작성 시 상위 후보의 profile heatmap과 이탈률 차이를 함께 확인한다.

In [5]:
selection_df = result_df.copy()

# 지나치게 작거나 큰 cluster가 있는 후보는 제외 
selection_df = selection_df[
    (selection_df["min_cluster_ratio"] >= 0.04) &
    (selection_df["max_cluster_ratio"] <= 0.45)
].copy()

if selection_df.empty:
    selection_df = result_df.copy()

# -----------------------------
# Internal validation ranks
# 낮을수록 좋은 rank score
# -----------------------------
internal_rank_specs = {
    "rank_silhouette": ("silhouette", False),
    "rank_davies_bouldin": ("davies_bouldin", True),
    "rank_calinski_harabasz": ("calinski_harabasz", False),
    "rank_min_cluster_ratio": ("min_cluster_ratio", False),
}

for rank_col, (metric_col, ascending) in internal_rank_specs.items():
    selection_df[rank_col] = selection_df[metric_col].rank(ascending=ascending, pct=True)

selection_df["internal_rank_score"] = (
    0.40 * selection_df["rank_silhouette"] +
    0.25 * selection_df["rank_davies_bouldin"] +
    0.20 * selection_df["rank_calinski_harabasz"] +
    0.15 * selection_df["rank_min_cluster_ratio"] 
)

# -----------------------------
# Business validation ranks
# 낮을수록 좋은 rank score
# 단, clustering 품질의 독립 검증/해석 보조 지표로 사용
# -----------------------------
business_rank_specs = {
    "rank_attrition_range": ("attrition_rate_range", False),
    "rank_risk_range": ("high_risk_signal_range", False),
}

for rank_col, (metric_col, ascending) in business_rank_specs.items():
    selection_df[rank_col] = selection_df[metric_col].rank(
        ascending=ascending,
        pct=True
    )

selection_df["business_rank_score"] = (
    0.60 * selection_df["rank_attrition_range"] +
    0.40 * selection_df["rank_risk_range"]
)

# -----------------------------
# 최종 종합 점수
# 내부 지표를 중심으로 하고, business 지표는 보조적으로만 반영
# 낮을수록 좋음
# -----------------------------
selection_df["overall_rank_score"] = (
    0.75 * selection_df["internal_rank_score"] +
    0.25 * selection_df["business_rank_score"]
)

# ============================================================
# Final model decision
# ============================================================
# 자동 rank score는 후보 비교용으로 유지하되,
# Step4.5 시각화/해석 결과를 반영하여 최종 모델은 명시적으로 고정한다.
# 최종 채택 모델: core_7_KMeans_k5
SELECTED_KEY_OVERRIDE = "core_7_KMeans_k5"
FINAL_SELECTED_KEY = SELECTED_KEY_OVERRIDE

# k=5 기준 segment 이름.
# 주의: cluster 번호는 같은 random_state/n_init/데이터에서 재현된다.
# n_init, sklearn version, 입력 데이터가 바뀌면 cluster 번호가 달라질 수 있으므로
# Step4.5 heatmap과 이탈률 그래프를 다시 확인해야 한다.
SEGMENT_NAME_MAP_K5 = {
    0: "상품 관계 중심 저위험 안정군",
    1: "신용 활용 중심 저위험 금융활용군",
    2: "고활동·저관계 잠재 전환군",
    3: "거래 감소·저활동·접촉 부담형 초고위험 이탈군",
    4: "저활동·비감소형 초저위험 안정군",
}

if SELECTED_KEY_OVERRIDE is not None:
    selected_key = SELECTED_KEY_OVERRIDE
    selection_method = (
        "manual_override_after_visual_diagnostics: "
        "해석 가능성, 이탈률 차이, cluster size 균형, k=4/k=6 비교를 기준으로 "
        "core_7_KMeans_k5를 최종 선택"
    )
else:
    selected_key = selection_df.sort_values(["overall_rank_score", "internal_rank_score"]).iloc[0]["key"]
    selection_method = "ranked_composite_score"

if selected_key not in all_labels:
    raise ValueError(f"선택된 key가 후보 label에 없습니다: {selected_key}")

if selected_key not in set(result_df["key"]):
    raise ValueError(f"선택된 key가 result_df에 없습니다: {selected_key}")

selected_row = result_df[result_df["key"] == selected_key].iloc[0].to_dict()
selected_labels = all_labels[selected_key]

selection_df["is_final_selected_model"] = selection_df["key"].eq(selected_key)

save_csv(
    selection_df.sort_values(["overall_rank_score", "internal_rank_score"]),
    output_dir / "03_model_selection_score.csv"
)

save_csv(
    result_df.sort_values(["silhouette", "attrition_rate_range"], ascending=[False, False]),
    output_dir / "04_candidate_metrics_sorted.csv"
)

# 최종 선택 모델이 포함된 비교표를 별도 저장
final_model_comparison = selection_df[
    selection_df["key"].isin(["core_7_KMeans_k4", "core_7_KMeans_k5", "core_7_KMeans_k6"])
].copy()
save_csv(
    final_model_comparison.sort_values(["k"]),
    output_dir / "04_final_model_comparison_core7_kmeans_k456.csv"
)

display(
    selection_df.sort_values(["overall_rank_score", "internal_rank_score"]).head(10)
)

print("Selected model:", selected_key)
print("Selection method:", selection_method)

,key,feature_set,algorithm,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_ratio,...,rank_silhouette,rank_davies_bouldin,rank_calinski_harabasz,rank_min_cluster_ratio,internal_rank_score,rank_attrition_range,rank_risk_range,business_rank_score,overall_rank_score,is_final_selected_model
0,core_7_KMeans_k4,core_7,KMeans,4,0.155258,1.814115,1545.182352,1938,3218,0.191370,...,0.066667,0.133333,0.066667,0.066667,0.083333,1.000000,0.933333,0.973333,0.305833,False
3,core_7_KMeans_k5,core_7,KMeans,5,0.144931,1.837940,1441.379165,998,2373,0.098548,...,0.200000,0.200000,0.200000,0.466667,0.240000,0.466667,0.733333,0.573333,0.323333,True
6,core_7_KMeans_k6,core_7,KMeans,6,0.138869,1.724660,1346.529889,870,2069,0.085909,...,0.400000,0.066667,0.333333,0.800000,0.363333,0.266667,0.266667,0.266667,0.339167,False
12,core_7_plus_capacity_KMeans_k5,core_7_plus_capacity,KMeans,5,0.144295,1.933031,1364.892013,937,2706,0.092525,...,0.266667,0.400000,0.266667,0.666667,0.360000,0.333333,0.333333,0.333333,0.353333,False
15,core_7_plus_capacity_KMeans_k6,core_7_plus_capacity,KMeans,6,0.139443,1.866946,1251.528426,836,2461,0.082552,...,0.333333,0.266667,0.400000,1.000000,0.430000,0.200000,0.066667,0.146667,0.359167,False
9,core_7_plus_capacity_KMeans_k4,core_7_plus_capacity,KMeans,4,0.150722,1.867298,1491.606070,1830,3151,0.180705,...,0.133333,0.333333,0.133333,0.133333,0.183333,0.933333,1.000000,0.960000,0.377500,False
14,core_7_plus_capacity_Agglomerative_Ward_k5,core_7_plus_capacity,Agglomerative_Ward,5,0.092854,2.300684,978.449360,971,3266,0.095882,...,0.466667,0.600000,0.533333,0.566667,0.528333,0.133333,0.166667,0.146667,0.432917,False
17,core_7_plus_capacity_Agglomerative_Ward_k6,core_7_plus_capacity,Agglomerative_Ward,6,0.075977,2.286869,920.422585,971,3172,0.095882,...,0.666667,0.533333,0.600000,0.566667,0.605000,0.066667,0.166667,0.106667,0.480417,False
5,core_7_Agglomerative_Ward_k5,core_7,Agglomerative_Ward,5,0.091023,2.152102,1051.093457,843,3261,0.083243,...,0.600000,0.466667,0.466667,0.933333,0.590000,0.400000,0.800000,0.560000,0.582500,False
10,core_7_plus_capacity_GMM_k4,core_7_plus_capacity,GMM,4,0.092032,3.154179,850.318817,1477,3332,0.145848,...,0.533333,0.800000,0.666667,0.200000,0.576667,0.866667,0.866667,0.866667,0.649167,False


Selected model: core_7_KMeans_k5
Selection method: manual_override_after_visual_diagnostics: 해석 가능성, 이탈률 차이, cluster size 균형, k=4/k=6 비교를 기준으로 core_7_KMeans_k5를 최종 선택


In [6]:
# ============================================================
# Review Shortlist 저장
# 목적:
# - 자동 점수 기준 최종 후보와 별개로,
# - core_7 + KMeans의 k=4,5,6 후보를 시각화 및 profiling 검토 대상으로 저장한다.
# - 최종 segment 수는 Step4.5 시각화와 Step5 profiling을 확인한 뒤 결정한다.
# ============================================================

REVIEW_FEATURE_SET = "core_7"
REVIEW_ALGORITHM = "KMeans"
REVIEW_K_VALUES = [4, 5, 6]

REVIEW_KEYS = [
    f"{REVIEW_FEATURE_SET}_{REVIEW_ALGORITHM}_k{k}"
    for k in REVIEW_K_VALUES
]

missing_review_keys = [key for key in REVIEW_KEYS if key not in all_labels]

if missing_review_keys:
    raise ValueError(f"review shortlist 후보가 all_labels에 없습니다: {missing_review_keys}")

review_selection_df = selection_df[
    selection_df["key"].isin(REVIEW_KEYS)
].copy()

review_selection_df = review_selection_df.sort_values(
    ["internal_rank_score", "business_rank_score"]
)

save_csv(
    review_selection_df,
    output_dir / "05_review_shortlist_core7_kmeans_k456.csv"
)

# 후보별 label 저장
review_label_df = pd.DataFrame({
    "Row_ID": df_cluster["Row_ID"].values
})

for key in REVIEW_KEYS:
    review_label_df[key] = all_labels[key]

save_csv(
    review_label_df,
    output_dir / "06_review_shortlist_cluster_labels.csv"
)

# review config 저장
review_config = {
    "purpose": "최종 segment 수 판단을 위해 core_7 KMeans k=4,5,6 후보를 저장한다.",
    "review_feature_set": REVIEW_FEATURE_SET,
    "review_algorithm": REVIEW_ALGORITHM,
    "review_k_values": REVIEW_K_VALUES,
    "review_keys": REVIEW_KEYS,
    "final_selected_key": selected_key,
    "selection_method": selection_method,
    "reproducibility_config": REPRODUCIBILITY_CONFIG,
    "note": (
        "cluster profile heatmap, 이탈률, cluster size, high-risk signal count, "
        "해석 가능성을 k=4,5,6 후보 간 비교한 뒤 최종 모델을 core_7_KMeans_k5로 고정한다."
    )
}

save_json(
    review_config,
    output_dir / "07_review_shortlist_config.json"
)

display(review_selection_df)
print("Review shortlist keys:")
for key in REVIEW_KEYS:
    print("-", key)

print("Final selected key:", selected_key)

,key,feature_set,algorithm,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size,max_cluster_size,min_cluster_ratio,...,rank_silhouette,rank_davies_bouldin,rank_calinski_harabasz,rank_min_cluster_ratio,internal_rank_score,rank_attrition_range,rank_risk_range,business_rank_score,overall_rank_score,is_final_selected_model
0,core_7_KMeans_k4,core_7,KMeans,4,0.155258,1.814115,1545.182352,1938,3218,0.191370,...,0.066667,0.133333,0.066667,0.066667,0.083333,1.000000,0.933333,0.973333,0.305833,False
3,core_7_KMeans_k5,core_7,KMeans,5,0.144931,1.837940,1441.379165,998,2373,0.098548,...,0.200000,0.200000,0.200000,0.466667,0.240000,0.466667,0.733333,0.573333,0.323333,True
6,core_7_KMeans_k6,core_7,KMeans,6,0.138869,1.724660,1346.529889,870,2069,0.085909,...,0.400000,0.066667,0.333333,0.800000,0.363333,0.266667,0.266667,0.266667,0.339167,False


Review shortlist keys:
- core_7_KMeans_k4
- core_7_KMeans_k5
- core_7_KMeans_k6
Final selected key: core_7_KMeans_k5


## 5. 선택 모델 산출물 저장

Step 4.5와 Step 5에서는 선택 모델 정보와 cluster label을 불러와 사용한다.  
현재 최종 선택 모델은 `core_7_KMeans_k5`로 고정하며, Step5에서 바로 사용할 최종 segment table도 함께 저장한다.


In [7]:
labels_df = pd.DataFrame({"Row_ID": df_cluster["Row_ID"]})
for key, labels in all_labels.items():
    labels_df[key] = labels
save_csv(labels_df, output_dir / "05_all_candidate_cluster_labels.csv")

selected_label_df = pd.DataFrame({
    "Row_ID": df_cluster["Row_ID"],
    "Cluster": selected_labels,
})
selected_label_df["Cluster_Label"] = selected_label_df["Cluster"].map(lambda x: f"C{int(x)}")

# 최종 모델이 k=5일 때 segment name 부여
if selected_key == "core_7_KMeans_k5":
    selected_label_df["Segment_Name"] = selected_label_df["Cluster"].map(SEGMENT_NAME_MAP_K5)
else:
    selected_label_df["Segment_Name"] = selected_label_df["Cluster_Label"]

save_csv(selected_label_df, output_dir / "10_selected_cluster_labels.csv")

selected_profile_summary = profile_summary_long[profile_summary_long["candidate_key"] == selected_key].copy()
if selected_key == "core_7_KMeans_k5":
    selected_profile_summary["Segment_Name"] = selected_profile_summary["Cluster"].map(SEGMENT_NAME_MAP_K5)
save_csv(selected_profile_summary, output_dir / "10_selected_model_profile_summary.csv")

# Step5에서 바로 사용할 최종 segment table 저장
final_segment_table = df_cluster.copy()
final_segment_table["Cluster"] = selected_labels
final_segment_table["Cluster_Label"] = final_segment_table["Cluster"].map(lambda x: f"C{int(x)}")
if selected_key == "core_7_KMeans_k5":
    final_segment_table["Segment_Name"] = final_segment_table["Cluster"].map(SEGMENT_NAME_MAP_K5)
else:
    final_segment_table["Segment_Name"] = final_segment_table["Cluster_Label"]

# profile table의 주요 해석/타깃 컬럼 연결
profile_attach_cols = [
    "Attrition_Flag",
    "Attrition_Label",
    "High_Risk_Signal_Count",
    "Strong_Decline_Flag",
    "Decline_Low_Activity_Flag",
    "High_Inactive_Flag",
    "High_Contact_Flag",
    "Low_Bank_Product_Count_Flag",
    "Zero_Utilization_Flag",
    "Total_Trans_Ct",
    "Total_Trans_Amt",
    "Total_Ct_Chng_Q4_Q1",
    "Total_Amt_Chng_Q4_Q1",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Relationship_Count",
    "Avg_Utilization_Ratio",
]

for col in profile_attach_cols:
    if col in df_profile.columns and col not in final_segment_table.columns:
        final_segment_table[col] = df_profile[col].values

save_csv(final_segment_table, output_dir / "13_final_customer_segments_core7_kmeans_k5.csv")

selected_model_info = {
    "selected_key": selected_key,
    "selected_row": selected_row,
    "feature_set": selected_row["feature_set"],
    "feature_columns": feature_sets[selected_row["feature_set"]],
    "algorithm": selected_row["algorithm"],
    "k": int(selected_row["k"]),
    "selection_method": selection_method,
    "manual_override": SELECTED_KEY_OVERRIDE is not None,
    "segment_name_map": SEGMENT_NAME_MAP_K5 if selected_key == "core_7_KMeans_k5" else None,
    "reproducibility_config": REPRODUCIBILITY_CONFIG,
    "important_note": (
        "Cluster IDs are reproducible only under the same input data, preprocessing, "
        "scikit-learn version, RANDOM_STATE, and n_init settings. "
        "If these change, re-run Step4.5 and verify segment names."
    ),
}
save_json(selected_model_info, output_dir / "09_selected_model.json")

# final selected model json을 명확한 파일명으로 한 번 더 저장
save_json(selected_model_info, output_dir / "09_final_selected_model_core7_kmeans_k5.json")

if selected_key in trained_models:
    joblib.dump(trained_models[selected_key], output_dir / "selected_clustering_model.joblib")
    joblib.dump(trained_models[selected_key], output_dir / "selected_clustering_model_core7_kmeans_k5.joblib")

save_csv(df_cluster, output_dir / "11_df_cluster_input_used.csv")
save_csv(df_profile, output_dir / "12_df_profile_table_used.csv")

run_metadata = {
    "n_rows": int(len(df_cluster)),
    "n_candidate_models": int(len(result_df)),
    "feature_sets": feature_sets,
    "algorithms": algorithms,
    "k_values": k_values,
    "selected_key": selected_key,
    "selection_method": selection_method,
    "reproducibility_config": REPRODUCIBILITY_CONFIG,
    "final_outputs": [
        "09_selected_model.json",
        "09_final_selected_model_core7_kmeans_k5.json",
        "10_selected_cluster_labels.csv",
        "10_selected_model_profile_summary.csv",
        "13_final_customer_segments_core7_kmeans_k5.csv",
        "selected_clustering_model_core7_kmeans_k5.joblib",
    ],
}
save_json(run_metadata, output_dir / "00_clustering_run_metadata.json")

print("Step 4 completed. Results saved to:", output_dir.relative_to(ROOT))
print("Final selected model:", selected_key)
print("Step5 input recommendation:", (output_dir / "13_final_customer_segments_core7_kmeans_k5.csv").relative_to(ROOT))

Step 4 completed. Results saved to: data\analysis\clustering_results
Final selected model: core_7_KMeans_k5
Step5 input recommendation: data\analysis\clustering_results\13_final_customer_segments_core7_kmeans_k5.csv
